<a href="https://colab.research.google.com/github/Ace2932/Fennec/blob/main/sim/nova_mjx/colab/fennec_distill.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Fennec — teacher → blind student distillation (#304)

`fennec_train.ipynb` produces a **teacher**. This produces the thing that actually ships.

Every checkpoint on Drive is a privileged teacher (obs 226/234, including a *perfect* 11×11
heightmap the real D456/L2 cannot supply). **No blind 105-d student exists yet**, so `policy_node`
has nothing to run — the bridge, safety profile and IMU driver are all in place and pointed at an
artifact that has never been produced.

The harness (`../distill.py`) was verified end to end on CPU 2026-08-09. **The pipeline is not what
you are testing here — you are testing whether a blind student can reproduce the teacher.**

Flow: config → clone → deps → Drive+teacher → sanity → calibrate → run → eval both commands → judge.

Recipe + rationale: [`DISTILL.md`](https://github.com/Ace2932/Fennec/blob/main/sim/nova_mjx/colab/DISTILL.md)


## 1. Config — single source of truth (edit here, nowhere else)

| knob | what it does |
|---|---|
| `SCALE` | **the only number you normally touch.** 1 = the CPU-smoke defaults, 10 = first real run. Drives BC episodes, DAgger episodes and epochs together. |
| `SCALE_CAL` | size of the timing calibration run. Keep small — its only job is to measure the GPU rate. |
| `TEACHER_DRIVE` | where the teacher `.pkl` lives on Drive. `artifacts/policies/*` is **gitignored**, so the clone will NOT have it. |
| `OUT_STEM` | student output stem. **Point at Drive** — `distill.py` has no resume and no intermediate checkpoint, so a dropout loses a VM-local artifact. |
| `VX_TRAIN` / `VX_SECOND` | the two commands to evaluate. Never rely on the `--vx` default. |

⚠️ Re-run this cell FIRST after any restart — a restart wipes the variables and the working directory
(the clone on disk persists, so the clone cell is fast the second time).


In [ ]:
# ---- paths -----------------------------------------------------------------
DRIVE     = "/content/drive/MyDrive"
BRANCH    = "main"
TEACHER_DRIVE = f"{DRIVE}/nova_policy_hm234.pkl"   # the one with a MEASURED flat baseline
TEACHER   = "artifacts/policies/nova_policy_hm234.pkl"   # where the run reads it from
OUT_STEM  = f"{DRIVE}/nova_student_blind"          # -> .pkl / .npz / .meta.json
LABEL     = "distill-v1"

# ---- scale -----------------------------------------------------------------
# distill.py's defaults ARE the CPU smoke (12/150 BC, 6/150 DAgger, 60 epochs).
# DISTILL.md: production is "10-100x the samples/epochs" — start at 10x, read the
# DAgger MSE trend, scale from EVIDENCE. Nobody has run this on GPU; do not guess.
SCALE      = 10
SCALE_CAL  = 2

BC_STEPS   = 150      # steps per episode — scale episodes, not steps
DAG_STEPS  = 150
_BC_EPS_1  = 12       # the 1x (smoke) sizes
_DAG_EPS_1 = 6
_EPOCHS_1  = 60

# ---- eval ------------------------------------------------------------------
VX_TRAIN   = 0.35     # the command with a measured teacher baseline (0% falls)
VX_SECOND  = 0.50     # distill.py's own default; teacher falls ~50% here
EVAL_EPISODES = 16
EVAL_STEPS    = 400

# ---- optimiser -------------------------------------------------------------
BATCH_SIZE = 256
LR         = 1e-3
SEED       = 0

# ---- derived (do not edit) --------------------------------------------------
def sizes(scale):
    return _BC_EPS_1*scale, _DAG_EPS_1*scale, _EPOCHS_1*scale

BC_EPISODES, DAG_EPISODES, EPOCHS = sizes(SCALE)
CAL_BC, CAL_DAG, CAL_EPOCHS       = sizes(SCALE_CAL)

print(f"run       {SCALE}x -> BC {BC_EPISODES}x{BC_STEPS} | DAgger {DAG_EPISODES}x{DAG_STEPS} | {EPOCHS} epochs")
print(f"calibrate {SCALE_CAL}x -> BC {CAL_BC}x{BC_STEPS} | DAgger {CAL_DAG}x{DAG_STEPS} | {CAL_EPOCHS} epochs")
print(f"eval      vx {VX_TRAIN} and {VX_SECOND}, {EVAL_EPISODES} paired episodes x {EVAL_STEPS} steps")
print(f"teacher   {TEACHER_DRIVE}")
print(f"student   {OUT_STEM}.pkl / .npz / .meta.json")


## 2. GPU + Python check


In [ ]:
import sys
print("Python", sys.version.split()[0])   # want 3.11/3.12
!nvidia-smi -L || echo "NO GPU -> Runtime > Change runtime type > GPU"


## 3. Get the code (fresh clone) — and REFUSE to run stale code

Fresh `rm -rf` + clone of the exact `BRANCH`, then assert the distillation markers exist. If the
harness is missing or older than #304, this raises rather than silently running something else.


In [ ]:
%cd /content
!rm -rf Fennec
!git clone --depth 1 -b {BRANCH} https://github.com/Ace2932/Fennec.git
%cd Fennec/sim/nova_mjx
!git -C /content/Fennec log --oneline -1
src = open("distill.py").read()
for marker in ("--eval-only", "export_student", "DAgger refit"):
    assert marker in src, f"distill.py is missing {marker!r} — wrong/stale branch"
print("distill.py OK")


## 4. Install pinned deps
brax 0.14.2 + jax 0.6.0 is the validated window. **If the sanity cell reports `cpu`:**
`Runtime -> Restart session`, then re-run **from cell 1**.


In [ ]:
!pip install -q "jax[cuda12]==0.6.0" brax==0.14.2 "orbax-checkpoint>=0.11.22" \
    "mujoco>=3.10" "mujoco-mjx>=3.10" "imageio>=2.31" imageio-ffmpeg 2>&1 | tail -3
print("deps installed — if the next cell says cpu, restart session + re-run from cell 1")


## 5. Mount Drive and put the teacher on the machine

⚠️ `sim/nova_mjx/artifacts/policies/*` is **gitignored** (`.gitignore:154`), so the fresh clone has
no checkpoints. Without this cell the run dies immediately. The file is ~1.4 MB.

**Record which teacher you used** — "the teacher" has meant four different files in this repo's history.


In [ ]:
from google.colab import drive
import os, shutil
drive.mount("/content/drive")
assert os.path.exists(TEACHER_DRIVE), f"teacher not on Drive: {TEACHER_DRIVE} — upload it or fix the path"
os.makedirs("artifacts/policies", exist_ok=True)
shutil.copy(TEACHER_DRIVE, TEACHER)
print("teacher:", TEACHER, os.path.getsize(TEACHER), "bytes")


## 6. Sanity — GPU backend (checked in a subprocess, not the kernel)

Keeps jax out of the notebook kernel so the later `!python` cells stay fork-safe.


In [ ]:
!python -c "import jax; b=jax.default_backend(); print('jax backend:', b); assert b=='gpu', 'not on GPU — restart session + re-run from cell 1'"


## 7. Calibrate before committing GPU-hours

Nobody has run this on a GPU, and `DISTILL.md` deliberately refuses to name a wall-clock. So measure
the rate at `SCALE_CAL` and extrapolate — the full run is roughly `SCALE / SCALE_CAL` times this.

This also smoke-tests the whole chain (collect → DAgger → fit → export → eval) on GPU before the
expensive run, and its output is written to a separate stem so it cannot clobber the real student.


In [ ]:
import time
t0 = time.time()
!python distill.py \
    --teacher {TEACHER} \
    --out {DRIVE}/nova_student_cal \
    --label cal-{SCALE_CAL}x \
    --bc-episodes {CAL_BC} --bc-steps {BC_STEPS} \
    --dagger-episodes {CAL_DAG} --dagger-steps {DAG_STEPS} \
    --epochs {CAL_EPOCHS} \
    --batch-size {BATCH_SIZE} --lr {LR} --seed {SEED} \
    --eval-episodes {EVAL_EPISODES} --eval-steps {EVAL_STEPS} \
    --vx {VX_TRAIN}
cal = time.time() - t0
print(f"\ncalibration {SCALE_CAL}x took {cal/60:.1f} min")
print(f"=> {SCALE}x projects to ~{cal/60*SCALE/SCALE_CAL:.0f} min "
      f"(~{cal/3600*SCALE/SCALE_CAL:.1f} h). Sanity-check that against your GPU budget BEFORE cell 8.")


## 8. The run

⚠️ **`--vx` is passed explicitly and is NOT the default.** `distill.py` defaults to `0.5`; the teacher
never falls at 0.35 and falls ~50 % of the time at 0.50. A student evaluated at 0.5 is being compared
against a teacher that is itself falling half the time. This project has already had to retract a
"teacher robustness is the blocker" conclusion reached on a fall rate quoted without its command.

⚠️ **No resume, no intermediate checkpoint.** The only write is at export. If Colab drops, the run is
lost — which is why `OUT_STEM` points at Drive and why you calibrated first.


In [ ]:
import time
t0 = time.time()
!python distill.py \
    --teacher {TEACHER} \
    --out {OUT_STEM} \
    --label {LABEL} \
    --bc-episodes {BC_EPISODES} --bc-steps {BC_STEPS} \
    --dagger-episodes {DAG_EPISODES} --dagger-steps {DAG_STEPS} \
    --epochs {EPOCHS} \
    --batch-size {BATCH_SIZE} --lr {LR} --seed {SEED} \
    --eval-episodes {EVAL_EPISODES} --eval-steps {EVAL_STEPS} \
    --vx {VX_TRAIN}
print(f"\nrun took {(time.time()-t0)/60:.1f} min")


## 9. Second command — eval only, no retrain

`--eval-only` reruns the paired eval against an already-exported student, so the second `vx` costs
an eval rather than a full distillation. **Never write a fall rate down without the `vx` beside it.**


In [ ]:
!python distill.py --eval-only {OUT_STEM}.pkl \
    --teacher {TEACHER} \
    --eval-episodes {EVAL_EPISODES} --eval-steps {EVAL_STEPS} \
    --vx {VX_SECOND}


## 10. Judge — read the output in THIS order

1. **`numpy vs Brax max|err|`**, printed at export. CPU verification got `5.25e-06`. If it is large,
   the `.npz` the robot would run is not the network that was trained and nothing below matters.
2. **Fall rate and speed TOGETHER, never separately.** A low fall rate at a speed well below command
   is a slow, conservative gait, not a better one — a student that learns to creep scores beautifully
   on falls.
3. **`return` is not comparable** teacher-vs-student; the teacher's reward carries swingref/gait/climb/PBRS
   terms the blind reward does not have. Compare fall-rate, distance, speed.

### Acceptance bar — the teacher's own paired numbers, not "better"

`nova_policy_hm234`, FLAT, 8 paired episodes × 400 steps, nominal dynamics:

| command | fell | speed | % of cmd | distance |
|---|---|---|---|---|
| **vx = +0.35** | **0.0 %** | +0.346 | 99 % | +2.215 m |

A student that falls where the teacher does not, or tracks materially below 99 % of command, has lost
information in the distillation.

### Kill-switch

If DAgger MSE has plateaued **and** the student still falls where the teacher does not at `vx 0.35`,
more epochs will not fix it — the blind observation is missing something the teacher was using. Stop
and look at *what* rather than spending GPU-hours. The 11×11 heightmap is the obvious suspect and the
whole reason a student is needed.

### What this run does NOT settle

**#309** (v7 swing-reference gate fails at shipped defaults) and **#311** (`cmd_c` sampled identically
for trot and crawl). The student inherits whatever the teacher learned under them — worth resolving
before a *retrain*, not before this run.


## 11. Keep the artifact + record what was run


In [ ]:
import json, os
meta = f"{OUT_STEM}.meta.json"
if os.path.exists(meta):
    print(json.dumps(json.load(open(meta)), indent=2))
else:
    print("no .meta.json — did the run reach export?")
!ls -la {OUT_STEM}.pkl {OUT_STEM}.npz {OUT_STEM}.meta.json 2>/dev/null
